In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

df = pd.read_csv("E:/Creditworthiness Tiering/corporate_rating.csv")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"




ratio_cols = [
    "currentRatio",
    "quickRatio",
    "cashRatio",
    "daysOfSalesOutstanding",
    "grossProfitMargin",
    "operatingProfitMargin",
    "pretaxProfitMargin",
    "netProfitMargin",
    "effectiveTaxRate",
    "returnOnAssets",
    "returnOnEquity",
    "returnOnCapitalEmployed",
    "debtRatio",
    "debtEquityRatio",
    "assetTurnover",
    "operatingCashFlowPerShare",
    "freeCashFlowPerShare",
    "cashPerShare",
    "operatingCashFlowSalesRatio",
    "freeCashFlowOperatingCashFlowRatio",
]



In [2]:

rating_col = [c for c in df.columns if "rating" in c.lower()][0]
sector_col = [c for c in df.columns if "sector" in c.lower()][0]

df[ratio_cols] = df[ratio_cols].replace([np.inf, -np.inf], np.nan)

rating_order = [
    "AAA",
    "AA",
    "A",
    "BBB",
    "BB",
    "B",
    "CCC",
    "CC",
    "C",
    "D",
]


def map_to_tier(rating):
    r = str(rating).upper().strip()
    if any(tier in r for tier in ["AAA", "AA", "A"]):
        return "1_High_Grade (AAA-A)"
    elif "BBB" in r:
        return "2_Medium_Grade (BBB)"
    elif any(tier in r for tier in ["BB", "B"]):
        return "3_Speculative (BB-B)"
    else:
        return "4_High_Distress (CCC-D)"


df["Credit_Tier"] = df[rating_col].apply(map_to_tier)
df["Is_Investment_Grade"] = df[rating_col].apply(
    lambda r: (
        "Investment Grade"
        if any(t in str(r).upper() for t in ["AAA", "AA", "A", "BBB"])
        else "High Yield / Junk"
    )
)



In [ ]:

print("=== 1. Rating Distribution Across Dataset ===")
print(df[rating_col].value_counts())

print("\n=== 2. Investment Grade vs. High Yield Breakdown ===")
print(df["Is_Investment_Grade"].value_counts(normalize=True) * 100)

overall_stats = df[ratio_cols].describe().T[["mean", "std", "50%", "25%", "75%"]]
overall_stats.columns = ["Mean", "Std_Dev", "Median", "Q1_25%", "Q3_75%"]
overall_stats.to_csv("descriptive_financial_ratios_summary.csv")
print("\n Saved 'descriptive_financial_ratios_summary.csv'")



key_ratios = [
    "currentRatio",
    "returnOnAssets",
    "returnOnEquity",
    "grossProfitMargin",
    "operatingProfitMargin",
    "debtRatio",
    "debtEquityRatio",
    "assetTurnover",
]

tier_medians = df.groupby("Credit_Tier")[key_ratios].median().T
tier_medians.to_csv("credit_tier_median_ratios.csv")
print(" Saved 'credit_tier_median_ratios.csv'")

print("\n=== 3. Median Financial Indicators Across Credit Tiers ===")
print(tier_medians.round(4))


plt.figure(figsize=(10, 5))
sns.countplot(
    data=df,
    x=rating_col,
    order=[
        r for r in rating_order if r in df[rating_col].astype(str).unique()
    ],
    palette="Blues_r",
    hue=rating_col,
    legend=False,
)
plt.title(
    "Distribution of Agency Credit Ratings (2010-2016)",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel("Credit Rating", fontsize=11)
plt.ylabel("Number of Companies", fontsize=11)
plt.tight_layout()
plt.savefig("eda_rating_distribution.png", dpi=300, bbox_inches="tight")
plt.close()


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ROA Profile
sns.boxplot(
    data=df,
    x="Credit_Tier",
    y="returnOnAssets",
    ax=axes[0, 0],
    palette="RdYlGn_r",
    hue="Credit_Tier",
    legend=False,
)
axes[0, 0].set_title("Return on Assets (ROA) by Credit Tier", fontweight="bold")
axes[0, 0].set_ylim(-0.10, 0.25)


sns.boxplot(
    data=df,
    x="Credit_Tier",
    y="debtRatio",
    ax=axes[0, 1],
    palette="RdYlGn_r",
    hue="Credit_Tier",
    legend=False,
)
axes[0, 1].set_title("Leverage (Debt Ratio) by Credit Tier", fontweight="bold")
axes[0, 1].set_ylim(0, 1.0)


sns.boxplot(
    data=df,
    x="Credit_Tier",
    y="currentRatio",
    ax=axes[1, 0],
    palette="RdYlGn_r",
    hue="Credit_Tier",
    legend=False,
)
axes[1, 0].set_title("Current Ratio by Credit Tier", fontweight="bold")
axes[1, 0].set_ylim(0, 4.0)


sns.boxplot(
    data=df,
    x="Credit_Tier",
    y="operatingProfitMargin",
    ax=axes[1, 1],
    palette="RdYlGn_r",
    hue="Credit_Tier",
    legend=False,
)
axes[1, 1].set_title(
    "Operating Profit Margin by Credit Tier", fontweight="bold"
)
axes[1, 1].set_ylim(-0.20, 0.50)

plt.tight_layout()
plt.savefig("eda_key_ratios_by_tier.png", dpi=300, bbox_inches="tight")
plt.close()


plt.figure(figsize=(12, 6))
sns.countplot(
    data=df,
    y=sector_col,
    hue="Is_Investment_Grade",
    palette=["#2b5c8f", "#d95f02"],
)
plt.title(
    "Investment Grade vs. High Yield Concentration by Sector",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel("Company Count", fontsize=11)
plt.ylabel("Industry Sector", fontsize=11)
plt.legend(title="Credit Status")
plt.tight_layout()
plt.savefig("eda_sector_credit_split.png", dpi=300, bbox_inches="tight")
plt.close()


plt.figure(figsize=(12, 9))
corr_matrix = df[key_ratios].corr(method="spearman")
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
)
plt.title(
    "Spearman Correlation Matrix of Core Financial Indicators",
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("eda_ratio_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.close()



=== 1. Rating Distribution Across Dataset ===
Rating
BBB    671
BB     490
A      398
B      302
AA      89
CCC     64
AAA      7
CC       5
C        2
D        1
Name: count, dtype: int64

=== 2. Investment Grade vs. High Yield Breakdown ===
Is_Investment_Grade
Investment Grade     57.417447
High Yield / Junk    42.582553
Name: proportion, dtype: float64

 Saved 'descriptive_financial_ratios_summary.csv'
 Saved 'credit_tier_median_ratios.csv'

=== 3. Median Financial Indicators Across Credit Tiers ===
Credit_Tier            1_High_Grade (AAA-A)  2_Medium_Grade (BBB)  \
currentRatio                         1.3587                1.4337   
returnOnAssets                       0.0721                0.0489   
returnOnEquity                       0.1714                0.1281   
grossProfitMargin                    0.5091                0.3850   
operatingProfitMargin                0.1435                0.1190   
debtRatio                            0.6121                0.6279   
debtEquit